# Homework: From *H. influenzae* annotation to protein evidence

Tyrone, the postdoc of the group ask you to run parsing and validation analysis on a dataset he was working on: The *Haemophilius influenzae* species. 

You received a pinned *Haemophilus influenzae* Rd KW20 reference, a bounded
GFF3 feature set, and the matching annotated protein FASTA. Extract selected
CDS sequences, translate them with the declared bacterial genetic code, and
compare your computed proteins with the supplied annotations.

Analyse at least six proteins and retain the evidence used to accept or question each computed translation. Tyrone tells you that the declared CDS translation table for H. influenzae is **11** and tips you to verify the dataset README and
GFF3 metadata rather than inferring conventions from filenames.


## Google Colab setup (optional)

Colab users should copy the repository folder to Google Drive, uncomment and
run the installation cell once per fresh runtime, and then uncomment the Drive
lines in the imports and paths cell below. Local Jupyter users can leave all
Colab-specific lines commented. Generated homework results stay under the
selected repository copy.


In [ ]:
# Google Colab only: uncomment and run once per fresh runtime.
# %pip install -q "biopython>=1.85,<2" "matplotlib>=3.9,<4" "pandas>=2.2,<3"


In [ ]:
from pathlib import Path
from urllib.parse import unquote

import matplotlib.pyplot as plt
import pandas as pd
from Bio import SeqIO
from Bio.Seq import Seq

REPO_ROOT = None

# Google Colab only: uncomment these lines after copying the repository to Drive.
# from google.colab import drive
# drive.mount("/content/drive")
# REPO_ROOT = Path("/content/drive/MyDrive/UIC_Bioinformatics_-_AI_for_Healthcare")

# Local Jupyter default: discover the repository from the current directory.
if "REPO_ROOT" not in globals():
    cwd = Path.cwd().resolve()
    REPO_ROOT = next((
        candidate for candidate in (cwd, *cwd.parents)
        if (candidate / "Module02_02" / "data").is_dir()
    ), None)

if REPO_ROOT is None:
    raise FileNotFoundError(
        "Repository not found. In Colab, mount Drive and set REPO_ROOT above."
    )

module_dir = Path(REPO_ROOT) / "Module02_02"
data_dir = module_dir / "data" / "teaching" / "haemophilus_influenzae"
fasta_path = data_dir / "reference.fna"
gff_path = data_dir / "subsets" / "homework_features.gff3"
protein_path = data_dir / "proteins.faa"
required_paths = [fasta_path, gff_path, protein_path]
missing_paths = [path for path in required_paths if not path.is_file()]
if missing_paths:
    raise FileNotFoundError(f"Missing required data files: {missing_paths}")
print("Module directory:", module_dir.resolve())


## Inspection step

Record the assembly and sequence identifiers, the GFF3 coordinate model,
strand and phase rules, and translation table 11. Passing a checksum proves
byte integrity, not agreement among reference, annotation, and proteins.


In [ ]:
references = {record.id: str(record.seq).upper() for record in SeqIO.parse(fasta_path, "fasta")}
annotated_proteins = {record.id: str(record.seq).rstrip("*") for record in SeqIO.parse(protein_path, "fasta")}
assert "NC_000907.1" in references
print("reference records:", len(references))
print("annotated proteins:", len(annotated_proteins))
print("reference length:", len(references["NC_000907.1"]))


## Parse attributes and group CDS rows

Parse the ninth field as semicolon-delimited `key=value` items. Group by
`protein_id`: repeated CDS rows can describe one biological product.


In [ ]:
def parse_gff3_attributes(text):
    attributes = {}
    for item in text.split(";"):
        if not item:
            continue
        key, value = item.split("=", 1) if "=" in item else (item, "")
        attributes[key] = unquote(value)
    return attributes

def group_cds_by_protein(path):
    # TODO: return one entry per protein_id with contig, strand, table, and
    # a list of (start, end, phase) segments.
    raise NotImplementedError

cds_by_protein = group_cds_by_protein(gff_path)
print("grouped proteins:", len(cds_by_protein))


## Coordinates, strand, and multiple segments

GFF3 start/end are 1-based closed; Python slices are 0-based half-open.
Order segments in biological 5-prime to 3-prime order, apply phase at each
segment's biological start, and reverse-complement negative-strand sequence
correctly.


In [ ]:
def reverse_complement(sequence):
    normalized = sequence.upper()
    unexpected = set(normalized) - set("ACGTN")
    if unexpected:
        raise ValueError(f"Unsupported symbols: {sorted(unexpected)}")
    return str(Seq(normalized).reverse_complement())

def extract_cds(reference, segments, strand):
    # TODO: validate bounds/strand/phase and join segments in biological order
    raise NotImplementedError


In [ ]:
assert reverse_complement("GATACA") == "TGTATC"
assert extract_cds("AACCGGTT", [(2, 5, 0)], "+") == "ACCG"
assert extract_cds("AACCGGTT", [(2, 5, 0)], "-") == "CGGT"
# Add lowercase, phase, invalid-coordinate, invalid-strand, repeated-ID,
# and missing-protein tests before using the real records.


## Translate and compare

Translate with table 11. Do not use `to_stop=True` to hide an internal stop.
Normalize only a terminal stop when the documented comparison requires it.
If a mismatch is limited to the first residue, inspect whether table 11
treats an alternative bacterial start as methionine under `cds=True`.
Use that mode only for a complete CDS with a valid terminal stop, record the
comparison rule, and never replace a residue by hand. Explain every
mismatch instead of silently repairing source data.


In [ ]:
rows = []
# TODO: select at least six proteins: two positive, two negative, one
# multi-row CDS, and one justified additional feature. Extract, translate
# with table 11, compare with annotated_proteins, and append evidence rows.
protein_validation = pd.DataFrame(rows)
protein_validation


In [ ]:
output_dir = module_dir / "results" / "homework"
output_dir.mkdir(parents=True, exist_ok=True)
protein_validation.to_csv(output_dir / "protein_validation.tsv", sep="	", index=False)

if not protein_validation.empty:
    ax = protein_validation.plot.bar(
        x="protein_id", y=["translated_length", "annotated_length"],
        figsize=(8, 3.5)
    )
    ax.set(ylabel="Amino acids", title="Computed versus annotated proteins")
    plt.tight_layout()
    plt.show()


## Interpret and submit

State assembly, coordinate, strand, phase, and table-11 assumptions. Report
one biological observation and one limitation. Complete `ai_use.md` based on the `assignment.md` instructions

Submit the notebook, `protein_validation.tsv`, `README.md`, and `ai_use.md`.